# ActionShap — fixed end-to-end experiment notebook

This notebook is the single execution interface for the corrected review response. It downloads/proves data provenance, runs the frozen two-dataset/two-model/five-seed suite, generates component result tables and figures, validates the manuscript, and performs reviewer-specific assertions.

**Important:** the final run is not a smoke test. It refuses to treat missing datasets, failed gates, missing users, or legacy schema-v1 outputs as evidence. Budget 1 and 3 are joint-action sensitivities only; they are not Actionability-Gap conditions. LOO is a deletion oracle, not a positive-gap competitor.

In [1]:
from pathlib import Path
import json, os, subprocess, sys

CODE = Path.cwd()
if CODE.name != "code":
    CODE = Path("paper-ideas/ActionShap/code").resolve()
assert (CODE / "configs/final.yaml").exists(), CODE
os.chdir(CODE)
print("Running from", CODE)


Running from /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/code


## 1. Environment and deterministic contract

The lock file is the reproducibility contract. The suite uses independent candidate, user, tie, model, and attribution seeds.

In [2]:
import yaml
config = yaml.safe_load((CODE / "configs/final.yaml").read_text())
assert config["seeds"] == [42,43,44,45,46]
assert config["budget"] == 2
assert config["utility"] == "target_margin"
assert config["n_max"] == 20
assert config["models"] == ["itemknn", "profile"]
print(json.dumps({k: config[k] for k in ("datasets","models","seeds","n_max","budget","utility")}, indent=2))


{
  "datasets": [
    {
      "name": "MovieLens-1M",
      "format": "ml1m",
      "path": "data/ml-1m/ratings.dat",
      "rating_threshold": 4.0
    },
    {
      "name": "Amazon-Digital-Music",
      "format": "csv",
      "path": "data/amazon-digital-music/interactions.csv",
      "user_column": "user",
      "item_column": "item",
      "timestamp_column": "timestamp",
      "rating_column": "rating",
      "rating_threshold": 4.0
    }
  ],
  "models": [
    "itemknn",
    "profile"
  ],
  "seeds": [
    42,
    43,
    44,
    45,
    46
  ],
  "n_max": 20,
  "budget": 2,
  "utility": "target_margin"
}


## 2. Data download and provenance

Review the dataset terms before setting `ACTIONSHAP_ACCEPT_TERMS=1`. The cell deliberately refuses silent downloads. Existing verified files are reused.

In [3]:
import os
ml = CODE / "data/ml-1m/ratings.dat"
amz = CODE / "data/amazon-digital-music/interactions.csv"
if not (ml.exists() and amz.exists()):
    if os.environ.get("ACTIONSHAP_ACCEPT_TERMS") != "1":
        raise RuntimeError("Missing final datasets. Set ACTIONSHAP_ACCEPT_TERMS=1 after reviewing source terms, then rerun this cell.")
    subprocess.run([sys.executable, "scripts/download_datasets.py", "--dataset", "all", "--accept-dataset-terms"], check=True)
assert ml.exists() and ml.stat().st_size > 0
assert amz.exists() and amz.stat().st_size > 0
print("datasets ready", ml.stat().st_size, amz.stat().st_size)


datasets ready 24594131 4255252


## 3. Frozen final suite

This executes convergence first, then the primary matrix, full-catalogue robustness, and predeclared sensitivities. Do not edit the YAML after this cell begins.

In [4]:
subprocess.run([sys.executable, "scripts/run_final_suite.py", "--config", "configs/final.yaml"], check=True)


/Users/mlouhichi/Desktop/actionShap/.venv/bin/python /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/code/scripts/run_convergence.py --ratings /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/code/data/ml-1m/ratings.dat --dataset-name MovieLens-1M --dataset-format ml1m --rating-threshold 4.0 --output /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/code/results/raw/convergence_movielens-1m_itemknn.json --model itemknn --utility target_margin --users 100 --evaluation-size 200 --n-max 20 --minimum-interactions 4 --epochs 10 --profile-samples-per-user 1 --profile-learning-rate 0.03 --profile-regularization 0.0001 --candidate-seed 1729 --user-seed 2718 --tie-seed 31415 --budgets 25,50,100,250,500,1000 --reference 1000
 permutations  reference_permutations  mean_rank_correlation_to_reference  std_rank_correlation_to_reference  valid_rank_seeds  mean_top1_agreement  mean_top2_jaccard  mean_top2_exact_agreement  mean_efficiency_e

CompletedProcess(args=['/Users/mlouhichi/Desktop/actionShap/.venv/bin/python', 'scripts/run_final_suite.py', '--config', 'configs/final.yaml'], returncode=0)

## 4. Generate corrected paper assets

The generator now writes:
- `aia_components.tex`: deletion AIA, bounded AIA, and gap for all five methods;
- `intervention_outcomes.tex`: effect, harm/success, abstention, and conditional regret;
- component figure with three panels;
- no budget rows in gap assets;
- no Shapley–LOO gap headline comparison.

In [9]:
subprocess.run([sys.executable, "scripts/make_paper_assets.py", "--raw", "results/schema-v2", "--out", "../paper"], check=True)
subprocess.run([sys.executable, "scripts/validate_manuscript.py", "--require-final"], check=True)


Python(47532) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


{
  "status": "PASS",
  "errors": [],
  "warnings": [],
  "notes": [
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s42.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s43.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s44.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s45.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s46.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_profile_s42.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_profile_s43.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue

Python(47572) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CompletedProcess(args=['/Users/mlouhichi/Desktop/actionShap/.venv/bin/python', 'scripts/validate_manuscript.py', '--require-final'], returncode=0)

## 5. Reviewer-specific scientific assertions

In [10]:
import pandas as pd
final = CODE.parent / "paper/final"
gap = pd.read_csv(final / "tables/actionability_gap_robustness.csv")
components = pd.read_csv(final / "tables/aia_components.csv")
assert set(gap.method) == {"shapley_mc","lime","loo","greedy_cf","random"}
assert not gap.condition_label.str.contains("B=1|B=3|budget", case=False, regex=True).any()
assert set(components.component) == {"Deletion AIA","Bounded AIA","Gap (bounded - deletion)"}
# Algebraic identity check at the method/condition summary level wherever finite.
for _, row in components.loc[components.component == "Gap (bounded - deletion)"].iterrows():
    sel = components[(components.dataset == row.dataset) & (components.model == row.model) & (components.evaluation_mode == row.evaluation_mode) & (components.condition == row.condition) & (components.method == row.method)]
    if len(sel) == 3:
        vals = dict(zip(sel.component, sel['mean']))
        assert abs(vals['Gap (bounded - deletion)'] - (vals['Bounded AIA'] - vals['Deletion AIA'])) < 1e-8
text = (CODE.parent / "paper/paper.tex").read_text()
for forbidden in ["only method with a positive", "22 comparisons", "uniquely intervention-robust", "Shapley alone improves"]:
    assert forbidden.lower() not in text.lower(), forbidden
assert "aia_components.tex" in text and "intervention_outcomes.tex" in text
print("review assertions passed", len(gap), "gap rows", len(components), "component rows")


review assertions passed 45 gap rows 135 component rows


## 6. Release checklist

A submission is allowed only if every item passes.

In [11]:
subprocess.run([sys.executable, "scripts/validate_review_contract.py", "--paper-root", "../paper"], check=True)


{'status': 'PASS', 'gap_rows': 45, 'component_rows': 135, 'errors': []}


Python(47595) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CompletedProcess(args=['/Users/mlouhichi/Desktop/actionShap/.venv/bin/python', 'scripts/validate_review_contract.py', '--paper-root', '../paper'], returncode=0)

In [12]:
report = json.loads((final / "manifests/validation_report.json").read_text())
assert report["status"] == "PASS", report
assert report["errors"] == []
print(json.dumps(report, indent=2))
print("READY: schema-v2 PASS, manuscript PASS, reviewer contradiction checks PASS")


{
  "status": "PASS",
  "errors": [],
  "warnings": [],
  "notes": [
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s42.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s43.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s44.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s45.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_itemknn_s46.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_profile_s42.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue_full_catalogue_profile_s43.json: secondary CSV lacks raw-source provenance (demo)",
    "r_Amazon_Digital_Music_full_catalogue

In [3]:
!python scripts/run_review5_experiments.py kernelschap --dataset movielens --users 1000 --ks-samples 256 512


[kernelschap] 50/1000 users
[kernelschap] 100/1000 users
[kernelschap] 150/1000 users
[kernelschap] 200/1000 users
[kernelschap] 250/1000 users
[kernelschap] 300/1000 users
[kernelschap] 350/1000 users
[kernelschap] 400/1000 users
[kernelschap] 450/1000 users
[kernelschap] 500/1000 users
[kernelschap] 550/1000 users
[kernelschap] 600/1000 users
[kernelschap] 650/1000 users
[kernelschap] 700/1000 users
[kernelschap] 750/1000 users
[kernelschap] 800/1000 users
[kernelschap] 850/1000 users
[kernelschap] 900/1000 users
[kernelschap] 950/1000 users
[kernelschap] 1000/1000 users
wrote results/review5/kernelschap_movielens.json


In [4]:
!python scripts/run_review5_experiments.py kernelschap --dataset amazon    --users 1000 --ks-samples 256 512


[kernelschap] 50/1000 users
[kernelschap] 100/1000 users
[kernelschap] 150/1000 users
[kernelschap] 200/1000 users
[kernelschap] 250/1000 users
[kernelschap] 300/1000 users
[kernelschap] 350/1000 users
[kernelschap] 400/1000 users
[kernelschap] 450/1000 users
[kernelschap] 500/1000 users
[kernelschap] 550/1000 users
[kernelschap] 600/1000 users
[kernelschap] 650/1000 users
[kernelschap] 700/1000 users
[kernelschap] 750/1000 users
[kernelschap] 800/1000 users
[kernelschap] 850/1000 users
[kernelschap] 900/1000 users
[kernelschap] 950/1000 users
[kernelschap] 1000/1000 users
wrote results/review5/kernelschap_amazon.json
